# Quantum simulation del trimero anello: Suzuki–Trotter con termine DM

Estensione a $N=3$ (anello isoscele) della quantum simulation già completata per il dimero. Riferimenti:

- **Teoria completa**: `quantum_simulation_trimero_anello_teoria.tex` — derivazione dei blocchi esatti, dell'operatore di chiralità di spin, e della struttura a **tre livelli** dell'errore di Trotter (non due, come nel dimero).
- **Applicazione**: `quantum_simulation_trimero_anello_applicazione.tex` — ricerca del punto di lavoro, convergenza in $N$, separazione dei contributi di errore, studio di compattezza del circuito.
- **Spiegazione passo-passo**: `quantum_simulation_trimero_anello_trotter_spiegato.tex` — questo stesso notebook, spiegato cella per cella.
- **Chiarimento**: `trimero_anello_frustrazione_e_chiralita.tex` — perché la non-commutatività dei legami non richiede frustrazione né equilatero.
- **Modulo**: `trotter_trimero_anello.py` — implementazione, quattro self-test.

Hamiltoniana (convenzione Pauli diretta, coerente con `trimer_ring_exact.py`):

$$H = b\sum_{i=1}^3 Z_i \;+\; \underbrace{J\,\vec\sigma_1\!\cdot\!\vec\sigma_2 + J'(\vec\sigma_2\!\cdot\!\vec\sigma_3+\vec\sigma_3\!\cdot\!\vec\sigma_1)}_{H_{ex}} \;+\; \underbrace{D_{12}(X_1Z_2{-}Z_1X_2)+D_{23}(X_2Z_3{-}Z_2X_3)+D_{31}(X_3Z_1{-}Z_3X_1)}_{H_{DM}}$$

con base $J$ sui siti $(1,2)$, lati $J'$ su $(2,3),(3,1)$, e Opzione B del DM ($D_{12}=rJ$, $D_{23}=D_{31}=rJ'$).

**Struttura a tre livelli** (derivata in teoria): $H_0=b\sum Z_i+H_{ex}$ fattorizza esattamente (campo↔scambio); ma sia $H_{ex}$ sia $H_{DM}$, essendo somme di tre legami che condividono i qubit a due a due, richiedono ciascuno un Trotter interno — due fonti di errore in più rispetto al dimero, non una sola.

**Nota su questa versione**: le Sezioni 4–5 (convergenza in $N$, separazione dei livelli di errore) sono ripetute su **tre** punti di lavoro robusti trovati dallo scan (non solo $R_0$), per verificare se le conclusioni quantitative sono specifiche del punto o generali — analogo al ventaglio di punti testati nel lavoro sul dimero.

## 0. Setup

In [ ]:
import numpy as np
import scipy.linalg as sla
import matplotlib.pyplot as plt
from qiskit.quantum_info import Statevector, Operator
from qiskit import QuantumCircuit

from trimer_ring_exact import trimer_hamiltonian, trimer_hamiltonian_dm, critical_field
from trotter_trimero_anello import (
    trotter_circuit, H_parts, SZ_TOT, PSI0, BONDS, _Jij,
    _bond_exchange_matrix, _bond_dm_matrix, step_Hex, step_HDM, step_field,
)

print("Qiskit e moduli caricati correttamente.")

## 1. Self-test del modulo

Quattro verifiche indipendenti, tutte a precisione macchina: (1) coerenza di $S_z^{tot}$ con il benchmark esatto; (2) il **circuito Qiskit reale** (non solo la costruzione a matrice) riproduce esattamente il prodotto atteso; (3) convergenza $N\to\infty$ verso l'evoluzione esatta; (4) fattorizzazione esatta di $H_0$, indipendente da $\tau$.

In [ ]:
from trotter_trimero_anello import _self_test
_self_test()

## 2. Ricerca del punto di lavoro

$J=1,J'=0.4$ fissi (punto VQE). Si scandisce una griglia $(b,D)$ cercando un regime con oscillazioni **non banali** di $\langle S_z^{tot}\rangle(t)$ da $\ket{000}$, con lo stesso criterio a due indicatori del dimero: escursione picco-picco e rapporto $a_2/a_1$ fra i due modi spettrali dominanti.

In [ ]:
J, Jp = 1.0, 0.4
bc = critical_field(J, Jp)
print(f"campo critico teorico (D=0): b_c = {bc}")

def mode_amplitudes(b, D):
    H = trimer_hamiltonian_dm(J, Jp, b, "B", D).to_matrix()
    E, V = np.linalg.eigh(H)
    c = V[0, :].conj()
    Sz_eig = V.conj().T @ SZ_TOT @ V
    amps = []
    for k in range(8):
        for l in range(k + 1, 8):
            Ckl = c[k].conjugate() * c[l] * Sz_eig[k, l]
            amps.append((2 * abs(Ckl), E[l] - E[k]))
    amps.sort(key=lambda x: -x[0])
    return amps

def sz_trace(b, D, t_max, n_t=1500):
    H = trimer_hamiltonian_dm(J, Jp, b, "B", D).to_matrix()
    E, V = np.linalg.eigh(H)
    c = V[0, :]
    ts = np.linspace(0, t_max, n_t)
    phase = np.exp(-1j * np.outer(ts, E))
    psi_t = (phase * c[None, :]) @ V.T
    sz = np.real(np.einsum('ti,ij,tj->t', psi_t.conj(), SZ_TOT, psi_t))
    return ts, sz

b_grid = np.linspace(0.05, 5.0, 60)
D_grid = np.linspace(0.05, 2.5, 40)
results = []
for b in b_grid:
    for D in D_grid:
        amps = mode_amplitudes(b, D)
        a1 = amps[0][0]
        a2 = amps[1][0] if len(amps) > 1 else 0.0
        ratio = a2 / a1 if a1 > 1e-9 else 0.0
        _, sz = sz_trace(b, D, t_max=30.0, n_t=1200)
        ptp = sz.max() - sz.min()
        results.append((b, D, ptp, ratio))
results = np.array(results)
print(f"scan completato: {len(results)} punti")

In [ ]:
mask = results[:, 2] > 0.5
cand = results[mask]
cand_sorted = cand[np.argsort(-cand[:, 3])]
print(f"{'b':>6} {'D':>6} {'D/J':>6} {'ptp':>8} {'a2/a1':>8}")
for row in cand_sorted[:8]:
    b, D, ptp, ratio = row
    print(f"{b:6.2f} {D:6.2f} {D / J:6.2f} {ptp:8.4f} {ratio:8.4f}")

**Verifica di robustezza** (lezione applicata dal dimero: non adottare il primo posto di un ranking cieco senza controllare che sia un plateau, non un punto fine-tuned).

In [ ]:
candidates = [(0.05, 1.93), (0.30, 1.31), (0.72, 1.24)]
for b0, D0 in candidates:
    print(f"\n--- attorno a (b={b0}, D={D0}) ---")
    for db, dD in [(0, 0), (0.05, 0), (-0.05, 0), (0, 0.1), (0, -0.1)]:
        b, D = b0 + db, D0 + dD
        if b <= 0:
            continue
        amps = mode_amplitudes(b, D)
        ratio = amps[1][0] / amps[0][0]
        _, sz = sz_trace(b, D, t_max=30.0, n_t=1200)
        ptp = sz.max() - sz.min()
        print(f"   b={b:5.2f} D={D:5.2f}: ptp={ptp:6.3f}  a2/a1={ratio:6.3f}")

## 3. Tre punti di lavoro adottati per il confronto

I tre candidati robusti trovati dallo scan, tutti con $J=1,J'=0.4$:

| nome | $b$ | $D$ | picco-picco | $a_2/a_1$ |
|---|---|---|---|---|
| $R_0$ (trimero) | 0.05 | 1.93 | 3.80 | 0.9998 |
| cand. 2 | 0.30 | 1.31 | 1.38 | 0.9982 |
| cand. 3 | 0.72 | 1.24 | 0.51 | 0.9992 |

$R_0$ resta il candidato principale (segnale maggiore, rapporto più vicino a 1); gli altri due sono usati qui **solo** per verificare la robustezza delle conclusioni quantitative delle Sezioni 4–5, non come alternative a pari titolo. **Tutti e tre restano provvisori**: nessuno discende da una derivazione fisica principiata come $R_1$ del dimero (rimandata a un notebook successivo).

In [ ]:
CANDIDATES = {
    "R0":     (0.05, 1.93),
    "cand_2": (0.30, 1.31),
    "cand_3": (0.72, 1.24),
}
COLORS = {"R0": "tab:blue", "cand_2": "tab:orange", "cand_3": "tab:green"}

# per ciascun punto: Hamiltoniana, autovalori/autovettori, periodo del modo dominante
punto_dati = {}
for nome, (b, D) in CANDIDATES.items():
    H = trimer_hamiltonian_dm(J, Jp, b, "B", D).to_matrix()
    E, V = np.linalg.eigh(H)
    c = V[0, :].conj()
    Sz_eig = V.conj().T @ SZ_TOT @ V
    amps = sorted(
        ((2 * abs(c[k].conjugate() * c[l] * Sz_eig[k, l]), E[l] - E[k])
         for k in range(8) for l in range(k + 1, 8)),
        key=lambda x: -x[0],
    )
    T_period = 2 * np.pi / amps[0][1]
    punto_dati[nome] = dict(b=b, D=D, H=H, E=E, V=V, c=V[0, :], T_period=T_period, amps=amps)
    print(f"{nome}: b={b}, D={D}  ->  periodo modo dominante = {T_period:.3f}")

## 4. Convergenza in $N$ — confronto sui tre punti

Per ciascun punto, traiettoria $\langle S_z^{tot}\rangle(t)$ su $T=2$ periodi \emph{del proprio} modo dominante (i tre punti hanno frequenze diverse), stato iniziale $\ket{000}$, confronto Trotter (circuito reale) contro dinamica esatta — **accanto a ciascun segnale, il suo spettro**: ampiezza dei modi di Bohr $a_{kl}=2|c_k^*c_l(S_z^{tot})_{kl}|$ in funzione del gap $\Delta_{kl}=E_l-E_k$ (formula ricavata in `quantum_simulation_trimero_anello_trotter_spiegato.tex` \S3.2.1), calcolato dalla stessa funzione `mode_amplitudes` usata per lo scan.

In [ ]:
psi0 = PSI0.data

def exact_traj(nome, ts):
    d = punto_dati[nome]
    phase = np.exp(-1j * np.outer(ts, d["E"]))
    psi_t = (phase * d["c"][None, :]) @ d["V"].T
    return np.real(np.einsum('ti,ij,tj->t', psi_t.conj(), SZ_TOT, psi_t))

def trotter_traj(nome, N):
    d = punto_dati[nome]
    T = 2 * d["T_period"]
    qc_step = trotter_circuit(J, Jp, d["b"], d["D"], T / N, 1)
    Ustep = Operator(qc_step).data
    psi = psi0.copy()
    ts = np.zeros(N + 1)
    sz = np.zeros(N + 1)
    sz[0] = float(np.real(psi.conj() @ SZ_TOT @ psi))
    for k in range(1, N + 1):
        psi = Ustep @ psi
        ts[k] = k * (T / N)
        sz[k] = float(np.real(psi.conj() @ SZ_TOT @ psi))
    return ts, sz

fig, axes = plt.subplots(3, 2, figsize=(14, 11))
for row, nome in enumerate(CANDIDATES):
    d = punto_dati[nome]
    ax_t, ax_f = axes[row, 0], axes[row, 1]
    T = 2 * d["T_period"]
    ts_fine = np.linspace(0, T, 1500)
    sz_fine = exact_traj(nome, ts_fine)
    ax_t.plot(ts_fine, sz_fine, color="0.15", lw=2.0, label="esatto")
    ts_N, sz_N = trotter_traj(nome, 160)
    ax_t.plot(ts_N, sz_N, "s", ms=4.0, mfc="none", mew=1.2, color=COLORS[nome],
               label="Trotter N=160", alpha=0.85)
    ax_t.set_title(nome + ": b=" + str(d["b"]) + ", D=" + str(d["D"]), fontsize=12)
    ax_t.set_ylabel(r"$\langle S_z^{tot}\rangle(t)$", fontsize=11)
    ax_t.grid(alpha=0.25)
    ax_t.legend(fontsize=9, loc="upper right")

    deltas = np.array([a[1] for a in d["amps"]])
    ampiezze = np.array([a[0] for a in d["amps"]])
    mask = ampiezze > 1e-3 * ampiezze.max()
    ax_f.stem(deltas[mask], ampiezze[mask], basefmt=" ", linefmt=COLORS[nome], markerfmt="o")
    ax_f.set_ylabel("ampiezza", fontsize=11)
    ax_f.set_title("spettro (modi di Bohr)", fontsize=11)
    ax_f.grid(alpha=0.25)

axes[-1, 0].set_xlabel("t (J=1)", fontsize=11)
axes[-1, 1].set_xlabel(r"$\Delta_{kl}$", fontsize=11)
fig.suptitle("Confronto delle dinamiche esatte ai tre punti di lavoro: segnale e spettro", y=1.00, fontsize=13)
fig.tight_layout()
plt.show()

Le tre dinamiche sono qualitativamente diverse: $R_0$ ha l'escursione maggiore e il periodo più lungo (campo debole, DM forte); il candidato 3 ha un'escursione molto più piccola e oscillazioni più rapide (campo forte, DM più debole in proporzione). Nessuna delle tre è un singolo coseno pulito, coerente col criterio $a_2/a_1$ usato per selezionarle.

In [ ]:
Ns = np.array([10, 20, 40, 80, 160, 320, 640, 1280])
infid_per_punto = {}

for nome in CANDIDATES:
    d = punto_dati[nome]
    T = 2 * d["T_period"]
    psi_ref_T = sla.expm(-1j * d["H"] * T) @ psi0
    infids = []
    for N in Ns:
        qc_step = trotter_circuit(J, Jp, d["b"], d["D"], T / N, 1)
        Ustep = Operator(qc_step).data
        psi = psi0.copy()
        for _ in range(N):
            psi = Ustep @ psi
        infids.append(1 - abs(np.vdot(psi_ref_T, psi)) ** 2)
    infid_per_punto[nome] = np.array(infids)

fig, ax = plt.subplots(figsize=(6.5, 5))
for nome in CANDIDATES:
    ax.loglog(Ns, infid_per_punto[nome], "o-", color=COLORS[nome], label=nome)
ref = infid_per_punto["R0"][3] * (Ns[3] / Ns) ** 2
ax.loglog(Ns, ref, "--", color="0.5", lw=1.2, label=r"riferimento $\propto 1/N^2$")
ax.set_xlabel("N (passi Trotter)")
ax.set_ylabel(r"infedeltà $1-|\langle\psi_{esatto}(T)|\psi_{Trotter}(T)\rangle|^2$")
ax.set_title("Convergenza in N, confronto sui tre punti")
ax.legend(fontsize=9)
ax.grid(alpha=0.25, which="both")
fig.tight_layout()
plt.show()

print(f"\n{'N':>6}", *[f"{nome:>14}" for nome in CANDIDATES])
for i, N in enumerate(Ns):
    print(f"{N:6d}", *[f"{infid_per_punto[nome][i]:14.3e}" for nome in CANDIDATES])

In [ ]:
print(f"{'punto':>8} {'<1%':>8} {'<0.1%':>8} {'<0.01%':>8}")
for nome in CANDIDATES:
    d = punto_dati[nome]
    T = 2 * d["T_period"]
    psi_ref_T = sla.expm(-1j * d["H"] * T) @ psi0
    thresholds = {}
    for target, label in [(1e-2, "1%"), (1e-3, "0.1%"), (1e-4, "0.01%")]:
        for N in range(10, 4000, 5):
            qc_step = trotter_circuit(J, Jp, d["b"], d["D"], T / N, 1)
            Ustep = Operator(qc_step).data
            psi = psi0.copy()
            for _ in range(N):
                psi = Ustep @ psi
            if 1 - abs(np.vdot(psi_ref_T, psi)) ** 2 < target:
                thresholds[label] = N
                break
    print(f"{nome:>8} {thresholds.get('1%','>4000'):>8} {thresholds.get('0.1%','>4000'):>8} {thresholds.get('0.01%','>4000'):>8}")

**Lettura del confronto**: la scala di $N$ richiesta varia sensibilmente da punto a punto (non è un numero universale). È atteso dalla teoria: l'errore di livello 1 scala con $J'^2$ (uguale ai tre punti, non spiega la differenza) mentre quello di livello 0 e 2 crescono con $D$ e con $b$ — punti con $D$ più grande (R0) o con $b$ più grande (cand. 3) pagano un prezzo diverso. Il punto qualitativamente robusto attraverso tutti e tre è lo **scaling** $O(1/N^2)$ (le pendenze in scala log-log sono le stesse), non il valore assoluto di $N$ necessario.

## 5. Separazione dei contributi di errore — confronto sui tre punti

Confronto fra il circuito **reale** ($V_2$: $H_{ex}$ *e* $H_{DM}$ spezzati nei bond) e una versione **idealizzata** — irrealizzabile a gate fisici — dove $H_{DM}$ è trattato come blocco esatto ($V_1$: solo livello 1). Ripetuto sui tre punti per vedere se il peso del livello 2 è specifico di $R_0$ o generale.

In [ ]:
def U_step_matrix(b, D, tau, hdm_exact=False):
    U_field = sla.expm(-1j * b * tau * SZ_TOT)
    U_hex = np.eye(8, dtype=complex)
    for (i, j) in BONDS:
        Jij = _Jij((i, j), J, Jp)
        U_hex = sla.expm(-1j * Jij * tau * _bond_exchange_matrix(i, j)) @ U_hex
    if hdm_exact:
        H_DM_full = sum(D * _Jij((i, j), J, Jp) * _bond_dm_matrix(i, j) for (i, j) in BONDS)
        U_hdm = sla.expm(-1j * tau * H_DM_full)
    else:
        U_hdm = np.eye(8, dtype=complex)
        for (i, j) in BONDS:
            Dij = D * _Jij((i, j), J, Jp)
            U_hdm = sla.expm(-1j * Dij * tau * _bond_dm_matrix(i, j)) @ U_hdm
    return U_hdm @ U_field @ U_hex

print(f"{'punto':>8} {'N':>6} {'infed. V2 (reale)':>20} {'infed. V1 (idealizz.)':>24} {'rapporto':>10}")
rapporti_finali = {}
for nome in CANDIDATES:
    d = punto_dati[nome]
    b, D = d["b"], d["D"]
    T = 2 * d["T_period"]
    psi_ref_T = sla.expm(-1j * d["H"] * T) @ psi0
    for N in [320, 640, 1280]:
        tau = T / N
        psi_v1, psi_v2 = psi0.copy(), psi0.copy()
        U1, U2 = U_step_matrix(b, D, tau, True), U_step_matrix(b, D, tau, False)
        for _ in range(N):
            psi_v1 = U1 @ psi_v1
            psi_v2 = U2 @ psi_v2
        infid_v1 = 1 - abs(np.vdot(psi_ref_T, psi_v1)) ** 2
        infid_v2 = 1 - abs(np.vdot(psi_ref_T, psi_v2)) ** 2
        rapporto = infid_v2 / infid_v1
        print(f"{nome:>8} {N:6d} {infid_v2:20.3e} {infid_v1:24.3e} {rapporto:10.3f}")
        rapporti_finali[(nome, N)] = rapporto

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.2))
Ns_sep = [320, 640, 1280]
for nome in CANDIDATES:
    vals = [rapporti_finali[(nome, N)] for N in Ns_sep]
    ax.plot(Ns_sep, vals, "o-", color=COLORS[nome], label=nome)
ax.axhline(1.0, color="0.6", ls=":", lw=1, label="nessun effetto (rapporto=1)")
ax.set_xscale("log")
ax.set_xlabel("N")
ax.set_ylabel(r"rapporto infedeltà $V_2/V_1$ (peso del livello 2)")
ax.set_title("Peso del livello 2 (bond-splitting di $H_{DM}$) sui tre punti")
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

**Lettura**: il rapporto $V_2/V_1$ è stabile in $N$ per ciascun punto (conferma che entrambi i livelli scalano allo stesso ordine $O(1/N^2)$, come previsto dalla teoria), ma il **valore** varia moltissimo da punto a punto: $\simeq4.4$ a $R_0$, $\simeq2.0$ al candidato 2, $\simeq1.04$ al candidato 3 — qui il livello 2 pesa solo il 4\% in più, quasi trascurabile. Non c'è quindi un "fattore universale" da citare per il peso del livello 2: dipende fortemente dal punto di lavoro (in particolare da $D$, che compare al quadrato nella formula teorica del livello 2), e va sempre ridichiarato per il punto specifico in uso. Coerente con la nota di cautela della teoria (\S8.6): la disuguaglianza triangolare fissa l'ordine, non il valore esatto della costante — e qui vediamo che quella costante può variare di un fattore $\sim4$ fra i tre punti testati.

## 6. Il circuito

Due passi Trotter espliciti (al punto $R_0$), per mostrare la struttura ripetuta (i barrier separano i passi). Ordine per passo: $H_{ex}$ (3 bond) → campo (esatto) → $H_{DM}$ (3 bond). Il conteggio gate non dipende dal punto di lavoro (dipende solo dalla struttura del circuito, non dai valori numerici di $b,D$), quindi non serve ripeterlo sui tre punti.

In [ ]:
b_R0, D_R0 = CANDIDATES["R0"]
qc_demo = trotter_circuit(J, Jp, b_R0, D_R0, t=1.0, N=2)
print(f"profondità: {qc_demo.depth()}, n. gate totali: {qc_demo.size()}")
print("conteggio:", dict(qc_demo.count_ops()))
qc_demo.draw(output="mpl", fold=-1, style={"name": "iqp"})

## 7. Esplorazione a parametri liberi — $J,J',b,D$ a scelta

**Nota metodologica**: la prima versione di questa sezione usava slider `ipywidgets` (`interact`). Si sono rivelati inaffidabili nell'ambiente reale (bloccati per minuti senza produrre output, non solo lenti — probabilmente un handshake dei widget che non si risolve, non un problema di velocità di calcolo). Sostituiti con lo stesso metodo già usato per i tre punti fissi: **chiamata diretta della funzione**, nessun meccanismo di widget in mezzo.

Per esplorare una combinazione $(J,J',b,D)$: modifica gli argomenti nella cella sotto ed eseguila di nuovo (Shift+Invio). Oltre a segnale e spettro (prima riga), la funzione ora calcola anche **convergenza in $N$** e **peso del livello 2** per quella combinazione specifica (seconda riga) — stesso metodo a matrice delle Sezioni 4–5, non il circuito Qiskit completo (già verificato identico, più veloce da ricalcolare). Costo aggiuntivo verificato: $\sim19$ ms di calcolo puro per chiamata, trascurabile rispetto al tempo di disegno della figura.

In [ ]:
def esplora(J=1.0, Jp=0.4, b=0.05, D=1.93, Ns=(10, 20, 40, 80, 160, 320, 640, 1280)):
    H = trimer_hamiltonian_dm(J, Jp, b, "B", D).to_matrix()
    E, V = np.linalg.eigh(H)
    c = V[0, :].conj()
    Sz_eig = V.conj().T @ SZ_TOT @ V
    amps = sorted(
        ((2 * abs(c[k].conjugate() * c[l] * Sz_eig[k, l]), E[l] - E[k])
         for k in range(8) for l in range(k + 1, 8)),
        key=lambda x: -x[0],
    )
    a1 = amps[0][0]
    a2 = amps[1][0] if len(amps) > 1 else 0.0
    ratio_a2a1 = a2 / a1 if a1 > 1e-9 else 0.0

    # --- segnale e spettro (dinamica esatta, come prima) ---
    ts = np.linspace(0, 30.0, 1200)
    phase = np.exp(-1j * np.outer(ts, E))
    psi_t = (phase * c[None, :]) @ V.T
    sz = np.real(np.einsum('ti,ij,tj->t', psi_t.conj(), SZ_TOT, psi_t))
    ptp = sz.max() - sz.min()

    deltas = np.array([a[1] for a in amps])
    ampiezze = np.array([a[0] for a in amps])
    soglia = 1e-3 * max(ampiezze.max(), 1e-9)
    mask = ampiezze > soglia

    # --- convergenza in N e peso del livello 2 (stesso metodo a matrice di Sez. 4-5) ---
    def U_step_free(tau, hdm_exact=False):
        U_field = sla.expm(-1j * b * tau * SZ_TOT)
        U_hex = np.eye(8, dtype=complex)
        for (i, j) in BONDS:
            Jij = _Jij((i, j), J, Jp)
            U_hex = sla.expm(-1j * Jij * tau * _bond_exchange_matrix(i, j)) @ U_hex
        if hdm_exact:
            H_DM_full = sum(D * _Jij((i, j), J, Jp) * _bond_dm_matrix(i, j) for (i, j) in BONDS)
            U_hdm = sla.expm(-1j * tau * H_DM_full)
        else:
            U_hdm = np.eye(8, dtype=complex)
            for (i, j) in BONDS:
                Dij = D * _Jij((i, j), J, Jp)
                U_hdm = sla.expm(-1j * Dij * tau * _bond_dm_matrix(i, j)) @ U_hdm
        return U_hdm @ U_field @ U_hex

    T_period = 2 * np.pi / amps[0][1] if abs(amps[0][1]) > 1e-9 else 30.0
    T_conv = 2 * T_period
    psi0 = PSI0.data
    psi_ref_T = sla.expm(-1j * H * T_conv) @ psi0

    infids = []
    for N in Ns:
        tau = T_conv / N
        U = U_step_free(tau, False)
        psi = psi0.copy()
        for _ in range(N):
            psi = U @ psi
        infids.append(1 - abs(np.vdot(psi_ref_T, psi)) ** 2)
    infids = np.array(infids)

    Ns_sep = [N for N in Ns if N >= 320] or list(Ns[-3:])
    rapporti = []
    for N in Ns_sep:
        tau = T_conv / N
        U1, U2 = U_step_free(tau, True), U_step_free(tau, False)
        psi1, psi2 = psi0.copy(), psi0.copy()
        for _ in range(N):
            psi1 = U1 @ psi1
            psi2 = U2 @ psi2
        i1 = 1 - abs(np.vdot(psi_ref_T, psi1)) ** 2
        i2 = 1 - abs(np.vdot(psi_ref_T, psi2)) ** 2
        rapporti.append(i2 / i1 if i1 > 1e-14 else float("nan"))

    # --- figura 2x2: segnale+spettro sopra, convergenza+livello2 sotto ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))
    ax1, ax2, ax3, ax4 = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

    ax1.plot(ts, sz, color="tab:blue", lw=1.3)
    ax1.set_xlabel("t")
    ax1.set_ylabel(r"$\langle S_z^{tot}\rangle(t)$")
    ax1.set_title("segnale")
    ax1.grid(alpha=0.25)

    ax2.stem(deltas[mask], ampiezze[mask], basefmt=" ")
    ax2.set_xlabel(r"$\Delta_{kl}$")
    ax2.set_ylabel("ampiezza")
    ax2.set_title("spettro (modi di Bohr)")
    ax2.grid(alpha=0.25)

    ax3.loglog(Ns, infids, "o-", color="tab:blue")
    mid = len(Ns) // 2
    ref = infids[mid] * (np.array(Ns)[mid] / np.array(Ns)) ** 2
    ax3.loglog(Ns, ref, "--", color="0.5", lw=1, label=r"riferimento $\propto1/N^2$")
    ax3.set_xlabel("N")
    ax3.set_ylabel("infedeltà")
    ax3.set_title("convergenza in N  (T=%.2f)" % T_conv)
    ax3.legend(fontsize=8)
    ax3.grid(alpha=0.25, which="both")

    ax4.plot(Ns_sep, rapporti, "o-", color="tab:red")
    ax4.axhline(1.0, color="0.6", ls=":", lw=1, label="nessun effetto")
    ax4.set_xscale("log")
    ax4.set_xlabel("N")
    ax4.set_ylabel(r"rapporto $V_2/V_1$")
    ax4.set_title("peso del livello 2")
    ax4.legend(fontsize=8)
    ax4.grid(alpha=0.25)

    fig.suptitle("J=%.2f  J'=%.2f  b=%.2f  D=%.2f" % (J, Jp, b, D), y=1.00, fontsize=13)
    fig.tight_layout()
    plt.show()

    print("picco-picco = %.4f     a2/a1 = %.4f" % (ptp, ratio_a2a1))
    print("infedeltà a N=%d: %.3e   |   peso livello 2 (V2/V1) a N=%d: %.3f"
          % (Ns[-1], infids[-1], Ns_sep[-1], rapporti[-1]))


# modifica questi valori ed esegui di nuovo la cella per esplorare altri punti
esplora(J=1.0, Jp=0.4, b=0.05, D=1.93)

In [ ]:
# un paio di esempi ulteriori (stessi J,J' del punto VQE, b,D diversi)
esplora(J=1.0, Jp=0.3, b=0.2, D=1.1)

## 8. Conclusioni e prossimi passi

- Modulo `trotter_trimero_anello.py` validato a precisione macchina (4 self-test, incluso il circuito Qiskit reale).
- Struttura a **tre livelli** confermata e derivata in forma chiusa: $H_0$ esatto, $H_{ex}$ e $H_{DM}$ entrambi con Trotter interno.
- Tre punti di lavoro testati per la convergenza e la separazione dei livelli di errore, non solo $R_0$:
  - la **scala di $N$** richiesta per una data soglia di infedeltà **varia da punto a punto** (non un numero universale), ma lo **scaling** $O(1/N^2)$ è confermato su tutti e tre;
  - il **peso del livello 2** varia moltissimo da punto a punto: da quasi trascurabile ($\sim1.04\times$, cand. 3) a dominante ($\sim4.4\times$, $R_0$) — non esiste un valore universale, va sempre misurato sul punto specifico in uso.
- $R_0$ (trimero) resta il punto principale per la dimostrazione, **provvisorio**: $J{=}1,J'{=}0.4,b{=}0.05,D{=}1.93$.

**Aperture**:
1. Derivazione R1-style del punto di lavoro (struttura degli 8 livelli sotto Opzione B) — rimandata, priorità più bassa delle correlazioni dinamiche.
2. Estensione al circuito con l'ancilla per le correlazioni dinamiche su $N=3$ — prossimo passo concordato col relatore.
3. Aggancio con la Parte 2 (rumore): costo in gate per passo (15 nativi) e scala $N$ richiesta, in relazione al vantaggio di gate di RBS su $W$ già documentato nel VQE.